In [ ]:
import numpy as np
import pandas as pd

In [14]:
class data_split():
    def __init__(self):
        pass

    def train_test_split(self, x, y, test_size=0.3):
        test_lenght = int(len(x) * test_size)
        train_lenght = len(x) - test_lenght

        x = pd.DataFrame(x)
        y = pd.Series(y)
        indexes = x.sample(frac=1, random_state=21).index       # Псевдослучайная выборка
        x = x.loc[indexes].reset_index(drop=True)
        y = y.loc[indexes].reset_index(drop=True)
        
        x_train = np.array(x.iloc[0:train_lenght])
        y_train = np.array(y.iloc[0:train_lenght])
        x_test = np.array(x.iloc[train_lenght:])
        y_test = np.array(y.iloc[train_lenght:])

        return x_train, x_test, y_train, y_test
    
    def train_test_val_split(self, x, y, validation_size=0.2, test_size=0.2): #return: x_train, x_validation, x_test, y_train, y_validation, y_test
        test_lenght = int(len(x) * test_size)
        validation_lenght = int(len(x) * validation_size)
        train_lenght = len(x) - test_lenght - validation_lenght

        x = pd.DataFrame(x)
        y = pd.Series(y)
        indexes = x.sample(frac=1, random_state=21).index       # Псевдослучайная выборка
        x = x.loc[indexes].reset_index(drop=True)
        y = y.loc[indexes].reset_index(drop=True)
        
        x_train = np.array(x.iloc[0:train_lenght])
        y_train = np.array(y.iloc[0:train_lenght])
        x_validation = np.array(x.iloc[train_lenght:(train_lenght+validation_lenght)])
        y_validation = np.array(y.iloc[train_lenght:(train_lenght+validation_lenght)])        
        x_test = np.array(x.iloc[(train_lenght+validation_lenght):])
        y_test = np.array(y.iloc[(train_lenght+validation_lenght):])

        return x_train, x_validation, x_test, y_train, y_validation, y_test


In [18]:
class data_time_split():
    def __init__(self):
        pass

    def train_test_split(self, x, y, column_name, date_split):
        date_split = pd.to_datetime(date_split, format="mixed", dayfirst=True)
        x = pd.DataFrame(x).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        if (column_name != None) and (column_name in x.columns):
            x[column_name] = pd.to_datetime(x[column_name], format="mixed", dayfirst=True)
            x = x.sort_values(by=column_name)
            x_train = x[x[column_name] < date_split]
            train_ind = x_train.index
            y_train = y.loc[train_ind]
            x_test = x[x[column_name] >= date_split]
            test_ind = x_test.index
            y_test = y.loc[test_ind]
            
            return x_train, x_test, y_train, y_test

    def train_test_val_split(self, x, y, column_name, validation_date, test_date):
        validation_date = pd.to_datetime(validation_date, format="mixed", dayfirst=True)
        test_date = pd.to_datetime(test_date, format="mixed", dayfirst=True)

        x = pd.DataFrame(x).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        if (column_name != None) and (column_name in x.columns):
            x[column_name] = pd.to_datetime(x[column_name], format="mixed", dayfirst=True)
            x = x.sort_values(by=column_name)
            x_train = x[x[column_name] < validation_date]
            train_ind = x_train.index
            y_train = y.loc[train_ind]

            x_validation = x[(x[column_name] >= validation_date) & (x[column_name] < test_date)]
            validation_ind = x_validation.index
            y_validation = y.loc[validation_ind]

            x_test = x[x[column_name] >= test_date]
            test_ind = x_test.index
            y_test = y.loc[test_ind]
            
            return x_train, x_validation, x_test, y_train, y_validation, y_test

# Cross-validation methods

### K-fold

In [23]:
class K_fold():
    def __init__(self):
        pass

    def split(self, x, folds=2):
        return_indexes = []
        # x = np.array(x)
        x = pd.DataFrame(x).reset_index(drop=True)
        fold_lenght = int(len(x) / folds)                       # Длина k - 1 фолдов
        # last_fold_lenght = len(x) - (fold_lenght * (folds - 1)) # Длина последнего фолда
        indexes = x.sample(frac=1, random_state=21).index       # Псевдослучайная выборка
        x = x.loc[indexes]          # Перемешаем данные
        for i in range(folds):
            begin = i * fold_lenght
            end = begin + fold_lenght
            #end_k = begin + last_fold_lenght
            if (i == folds - 1):
                test_fold = x.iloc[begin:].index
                train_fold = x.drop(index=test_fold, axis=0).index
                return_indexes.append([train_fold, test_fold])
            else:
                test_fold = x.iloc[begin:end].index
                train_fold = x.drop(index=test_fold, axis=0).index
                return_indexes.append([train_fold, test_fold])
        return return_indexes

### GroupKFold

In [31]:
class Group_K_fold():       # основная проблема реализации - при количестве фолдов меньшем, чем количество групп, одна или несколько групп никогда не будут учавствовать в тестовых наборах данных
    def __init__(self):
        pass

    def split(self, x, group_field, folds=2):           # group_field численный массив, каждое значение которого соответствует требуемой группе. Нумерация групп начинается с 0! 
        if len(x) != len(group_field):
            print("Параметры x и group_field должны быть одинаковой длины!")
            return None
        return_indexes = []
        x = pd.DataFrame(x).reset_index(drop=True)
        x = pd.concat([x, pd.Series(group_field, name='Group').reset_index(drop=True)], axis=1)
        groups = len(x["Group"].unique())                       # Количество групп
        if folds > groups:
            print("Параметр folds должен быть меньше или равен количеству групп в group_field")
            return None
        x = x.sort_values(by="Group")

        for i in range(folds):
            test_fold = x[x["Group"] == i].index
            train_fold = x[x["Group"] != i].index
            return_indexes.append([train_fold, test_fold])

        return return_indexes

### Stratified Kfold

In [72]:
class Stratified_K_fold():       
    def __init__(self):
        pass

    def split(self, x, y, folds=2):             # параметр y для задач классификации, при регрессии - это параметр соответствия примеров классам
        if len(x) != len(y):
            print("Аргументы x и y должны быть одинаковой длины!")
            return None
        
        return_indexes = []
        x = pd.DataFrame(x).reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        #x = pd.concat([x, pd.Series(y, name='Classes').reset_index(drop=True)], axis=1)

        classes = y.unique()                       # Количество групп
        quantity_of_clases = len(classes)               
        full_len = len(x)
        self.ratio = []
        df_list = []
        for i in range(quantity_of_clases):
            df_class = y.loc[y == classes[i]]     # Датафрейм отдельного класса 
            class_len = len(df_class)
            self.ratio.append(class_len / full_len)
            # df_list.append(df_class.sample(frac=1, random_state=21))
            df_list.append(df_class)
            
        #x = x.sort_values(by=["Classes"])

        fold_lenght = int(len(x) / folds)                       # Длина k - 1 фолдов
        # last_fold_lenght = len(x) - (fold_lenght * (folds - 1)) # Длина последнего фолда
        for i in range(folds):
            test_fold = []
            train_folds = []
            if i != folds - 1:
                for item in range(quantity_of_clases):
                    begin =  i * int(fold_lenght * self.ratio[item])
                    end = begin + int(fold_lenght * self.ratio[item])

                    test_indexes = df_list[item].iloc[begin:end].index
                    train_indexes = df_list[item].drop(index=test_indexes, axis=0).index
                    for k in test_indexes:
                        test_fold.append(k)
                    for k in train_indexes:
                        train_folds.append(k)
                return_indexes.append([train_folds, test_fold])

            else:
                for item in range(quantity_of_clases):
                    begin =  i * int(fold_lenght * self.ratio[item])

                    test_indexes = df_list[item].iloc[begin:].index
                    train_indexes = df_list[item].drop(index=test_indexes, axis=0).index
                    for k in test_indexes:
                        test_fold.append(k)
                    for k in train_indexes:
                        train_folds.append(k)
                return_indexes.append([train_folds, test_fold])

        return return_indexes